In [ ]:
# Système et fichiers
import os
import shutil
import sys
import tempfile
from pathlib import Path

# Manipulation de données
import pandas as pd

# API
from fastapi.testclient import TestClient

sys.path.insert(0, str(Path.cwd().parent))
from api.database import CHEMIN_BASE_DEFAUT
from api.main import app

pd.set_option("display.max_columns", None)


# API REST IRVE

L'API expose la base `data/database/irve_database.db` construite par `02_database_setup.ipynb`. Son code est dans le dossier `api/` ; ce notebook en montre le fonctionnement.

## Organisation

| Fichier | Rôle |
|---|---|
| `api/main.py` | Application FastAPI et routes |
| `api/database.py` | Connexion SQLite : une connexion par requête, clés étrangères activées, fermeture garantie même en cas d'erreur |
| `api/schemas.py` | Modèles Pydantic : validation des entrées, description des réponses |
| `api/security.py` | Contrôle de la clé d'API pour les opérations d'écriture |
| `api/tests/test_api.py` | Tests automatiques (pytest), exécutés sur une copie temporaire de la base |

## Routes

| Méthode | Route | Rôle | Accès |
|---|---|---|---|
| GET | `/stats` | Chiffres clés du réseau et des déserts | Public |
| GET | `/communes` | Communes filtrées par département, région, désert ; pagination | Public |
| GET | `/communes/{code_insee}` | Fiche d'une commune : équipement, points pour 10 000 habitants, distances | Public |
| GET | `/deserts` | Communes en désert (recharge ou 150 kW), les plus peuplées d'abord | Public |
| GET | `/stations`, `/stations/{id_station}` | Stations d'une commune ; fiche d'une station et de ses points | Public |
| GET | `/points-de-charge`, `/points-de-charge/{id_pdc}` | Points de charge filtrés par station ou puissance minimale | Public |
| POST | `/points-de-charge` | Créer un point de charge | Clé d'API |
| PUT | `/points-de-charge/{id_pdc}` | Remplacer toutes les caractéristiques d'un point | Clé d'API |
| PATCH | `/points-de-charge/{id_pdc}` | Modifier une partie des caractéristiques d'un point | Clé d'API |
| DELETE | `/points-de-charge/{id_pdc}` | Supprimer un point | Clé d'API |

## Sécurité

| Mesure | Détail |
|---|---|
| Authentification des écritures | En-tête `X-API-Key` exigé pour POST, PUT, PATCH et DELETE ; clé lue dans le fichier `.env` (exclu de Git) ou la variable d'environnement `IRVE_API_KEY`, jamais écrite dans le code ; comparaison en temps constant ; sans clé configurée, l'écriture est désactivée (503) |
| Validation des entrées | Puissance entre 0 (exclu) et 600 kW, prises booléennes, date de mise en service entre 2000 et aujourd'hui, identifiants au format attendu, champs inconnus refusés : réponse 422 avec le motif |
| Injection SQL | Toutes les valeurs reçues sont passées en paramètres des requêtes, jamais concaténées au texte SQL |
| Intégrité | Clés étrangères activées à chaque connexion ; station inexistante refusée (422) ; identifiant déjà utilisé refusé (409) ; contraintes de la base en dernier recours (409) |
| Volume | Pagination plafonnée à 500 lignes par appel |
| Exposition | Serveur local (127.0.0.1), aucune plateforme cloud |

## Pourquoi une API plutôt qu'un accès direct à la base ?

| Critère | Accès direct à SQLite | API |
|---|---|---|
| Droits | Quiconque a le fichier peut tout lire, modifier ou supprimer | Lecture publique, écriture réservée aux détenteurs de la clé |
| Qualité des données | Seules les contraintes de la base protègent | Règles métier vérifiées avant l'écriture, messages d'erreur explicites |
| Dépendance au schéma | Chaque utilisateur doit connaître tables et jointures | Routes stables : le schéma ou le moteur (SQLite vers PostgreSQL) peut changer sans casser les usages |
| Interopérabilité | Client SQLite nécessaire | HTTP et JSON, utilisables par un tableau de bord, un notebook ou une application |
| Documentation | Aucune | Documentation interactive générée automatiquement (`/docs`) |

## Lancer l'API

Depuis la racine du projet, environnement virtuel activé :

```bash
uvicorn api.main:app --host 127.0.0.1 --port 8003
```

Documentation interactive : http://127.0.0.1:8003/docs. Pour tester une écriture depuis cette page, cliquer sur **Authorize** et saisir la clé du fichier `.env`.

## Lancer les tests

Depuis la racine du projet :

```bash
python -m pytest -v
```


## Démonstration

Les appels ci-dessous passent par un client de test, sans lancer de serveur. Ils travaillent sur une **copie temporaire** de la base : la base réelle n'est pas modifiée.


In [ ]:
dossier_demo = Path(tempfile.mkdtemp())
base_demo = dossier_demo / "irve_demo.db"
shutil.copy(CHEMIN_BASE_DEFAUT, base_demo)
os.environ["IRVE_DB_PATH"] = str(base_demo)
os.environ["IRVE_API_KEY"] = "cle-de-demonstration"
ENTETE = {"X-API-Key": "cle-de-demonstration"}
client = TestClient(app)
print(f"Copie de travail : {base_demo}")


In [ ]:
print("GET /stats")
print(pd.Series(client.get("/stats").json()).to_string())

print("\nGET /deserts : communes en désert de recharge les plus peuplées")
deserts = pd.DataFrame(client.get("/deserts", params={"limit": 10}).json())
print(deserts[["code_insee", "nom_commune", "region", "population", "distance_pdc_km"]].to_string(index=False))

print("\nGET /communes/58194 (Nevers)")
print(pd.Series(client.get("/communes/58194").json()).to_string())


In [ ]:
station = client.get("/stations", params={"code_insee": "58194", "limit": 1}).json()[0]["id_station"]
point = {"id_pdc": "FRDEMOE0000001", "id_station": station, "puissance_kw": 150.0,
         "prise_type_2": False, "prise_type_combo_ccs": True, "prise_type_chademo": False,
         "prise_type_ef": False, "date_mise_en_service": "2025-06-30"}

reponse = client.post("/points-de-charge", json=point)
print("POST sans clé                  :", reponse.status_code, reponse.json()["detail"])

reponse = client.post("/points-de-charge", json=point | {"puissance_kw": 900}, headers=ENTETE)
print("POST puissance 900 kW          :", reponse.status_code, reponse.json()["detail"][0]["msg"])

reponse = client.post("/points-de-charge", json=point, headers=ENTETE)
print("POST valide                    :", reponse.status_code, reponse.json())

reponse = client.post("/points-de-charge", json=point, headers=ENTETE)
print("POST même identifiant          :", reponse.status_code, reponse.json()["detail"])

reponse = client.patch(f"/points-de-charge/{point['id_pdc']}", json={"puissance_kw": 300}, headers=ENTETE)
print("PATCH puissance 300 kW         :", reponse.status_code, reponse.json()["puissance_kw"])

reponse = client.put(f"/points-de-charge/{point['id_pdc']}",
                     json={k: v for k, v in point.items() if k != "id_pdc"} | {"prise_type_2": True},
                     headers=ENTETE)
print("PUT remplacement complet       :", reponse.status_code, reponse.json())

reponse = client.delete(f"/points-de-charge/{point['id_pdc']}", headers=ENTETE)
print("DELETE                         :", reponse.status_code)
print("GET après suppression          :", client.get(f"/points-de-charge/{point['id_pdc']}").status_code)

client.close()
shutil.rmtree(dossier_demo, ignore_errors=True)
